# Day 3. Linear Layer / Weight / Bias

**오늘의 목표**
- `y = xW + b`를 직접 계산하고, 각 기호가 무엇인지 설명할 수 있다
- Bias가 왜 필요한지 안다
- `nn.Linear(3, 5)`에서 3과 5의 의미, 그리고 내부 Weight의 shape를 안다
- Linear Layer가 Day 2의 `X @ Wq`와 같은 것이라는 걸 연결한다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)

## 1. `y = xW + b`

Linear Layer는 Day 2에서 배운 **행렬 곱**에 **덧셈 하나**를 더한 것입니다.

| 기호 | 이름 | 역할 | shape 예시 |
|---|---|---|---|
| `x` | 입력 | 들어오는 벡터 | `(3,)` |
| `W` | Weight (가중치) | 입력을 어떻게 섞을지 정하는 행렬 | `(3, 5)` |
| `b` | Bias (편향) | 결과에 더하는 값 | `(5,)` |
| `y` | 출력 | 새로운 벡터 | `(5,)` |

```text
x (3,)  @  W (3, 5)  →  (5,)
                  +  b (5,)
                  =  y (5,)
```

### 1-1. 손으로 따라가 보기

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])          # 입력: 숫자 3개

W = torch.tensor([[ 0.5, -1.0],             # (3, 2): 입력 3개 → 출력 2개
                  [ 1.0,  0.0],
                  [-0.5,  2.0]])
b = torch.tensor([0.1, -0.3])               # (2,)

xW = x @ W
y = xW + b

print("x @ W     :", xW)
print("x @ W + b :", y)

**출력 첫 번째 숫자가 어떻게 나왔는지** 직접 확인해보세요.

```text
y[0] = x · (W의 0번째 열) + b[0]
     = 1×0.5 + 2×1.0 + 3×(-0.5) + 0.1
     = 0.5 + 2.0 - 1.5 + 0.1
     = 1.1
```

즉 **출력 하나하나가 "입력 전체를 Weight 비율로 섞은 값"** 입니다.
W의 각 열은 "입력을 어떤 비율로 섞어서 출력 하나를 만들까"를 정하는 레시피예요.

## 2. Bias는 왜 필요한가?

Bias가 없으면 `y = xW`입니다. 이러면 **입력이 0일 때 출력도 반드시 0**이 됩니다.

In [ ]:
zero = torch.zeros(3)
print("Bias 없음 → zero @ W     =", zero @ W)
print("Bias 있음 → zero @ W + b =", zero @ W + b)

1차원으로 생각하면 더 쉽습니다.

```text
y = w·x        → 반드시 원점(0, 0)을 지나는 직선만 가능
y = w·x + b    → 위아래로 옮긴 직선도 가능
```

예를 들어 "섭씨 → 화씨" 변환은 `F = 1.8 × C + 32`입니다.
Bias(32) 없이는 Weight를 아무리 잘 골라도 이 관계를 표현할 수 없어요.

In [ ]:
celsius = torch.tensor([0.0, 100.0, 37.0])

w_only    = 1.8 * celsius          # bias 없음
with_bias = 1.8 * celsius + 32     # bias 있음

for c, a, b_ in zip(celsius, w_only, with_bias):
    print(f"{c.item():6.1f}°C  →  bias 없음: {a.item():6.1f}   bias 있음: {b_.item():6.1f}°F")

> 💡 **정리**: Weight는 "입력을 얼마나 키우고 섞을지", Bias는 "결과를 얼마나 옮길지"를 담당합니다.

## 3. PyTorch의 `nn.Linear`

### 3-1. 만들고 써보기

In [ ]:
linear = nn.Linear(3, 5)    # 입력 길이 3 → 출력 길이 5

x = torch.tensor([1.0, 2.0, 3.0])
y = linear(x)

print("입력 shape:", tuple(x.shape))
print("출력 shape:", tuple(y.shape))
print("출력:", y)

### 3-2. 안에 뭐가 들어있나? ⚠️ 주의할 점

In [ ]:
print("weight shape:", tuple(linear.weight.shape))
print("bias   shape:", tuple(linear.bias.shape))

**어? weight가 `(3, 5)`가 아니라 `(5, 3)`입니다!**

PyTorch는 Weight를 **(출력, 입력)** 순서로 저장하고, 계산할 때 뒤집어서 씁니다.

```text
nn.Linear 내부 계산:  y = x @ weight.T + bias
                         (3,) @ (3, 5) + (5,)
```

수식으로는 똑같이 `y = xW + b`이고, 저장 방식만 다릅니다. 코드를 읽을 때 헷갈리지 않도록 기억해두세요.

In [ ]:
manual = x @ linear.weight.T + linear.bias
print("nn.Linear 결과   :", linear(x))
print("직접 계산한 결과 :", manual)
print("같은가?", torch.allclose(linear(x), manual))

### 3-3. Weight는 누가 정하나?

방금 만든 `linear`의 Weight는 **랜덤 숫자**입니다. 아무 의미 없는 값이에요.

In [ ]:
print(linear.weight)

출력 끝에 `requires_grad=True`가 보이죠? "이 값은 **학습으로 바뀔 값**이다"라는 표시입니다.

```text
처음: 랜덤 Weight → 엉뚱한 출력
  ↓  정답과 비교해서 (Loss, Day 6)
  ↓  Weight를 조금씩 고침 (Backprop + Optimizer, Day 7)
나중: 쓸모 있는 Weight → 좋은 출력
```

**사람이 Weight를 정하지 않습니다. 데이터를 보고 모델이 스스로 찾아냅니다.**
이렇게 학습으로 찾는 값을 **파라미터(parameter)** 라고 부릅니다.

### 3-4. 맛보기: 정말 스스로 찾을까?

섭씨 → 화씨 데이터만 주고, `nn.Linear(1, 1)`이 `w=1.8`, `b=32`를 찾아내는지 봅시다.
학습 루프 코드는 Day 7에서 자세히 배우니, 지금은 **결과만** 보세요.

In [ ]:
C = torch.tensor([[-40.0], [0.0], [10.0], [37.0], [100.0]])
F = 1.8 * C + 32

model = nn.Linear(1, 1)
print(f"학습 전: w={model.weight.item():.3f}, b={model.bias.item():.3f}")

opt = torch.optim.Adam(model.parameters(), lr=0.5)
for step in range(3000):
    loss = ((model(C) - F) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()

print(f"학습 후: w={model.weight.item():.3f}, b={model.bias.item():.3f}   (정답: w=1.8, b=32)")

## 4. 파라미터 개수 세기

`nn.Linear(in, out)`의 파라미터 개수:

```text
weight: in × out 개
bias  : out 개
합계  : in × out + out
```

GPT-2 small이 1.24억 개, 큰 LLM이 수천억 개의 파라미터를 가진다고 할 때, 그 대부분이 바로 이런 Linear Layer의 Weight입니다.

In [ ]:
layer = nn.Linear(3, 5)
total = sum(p.numel() for p in layer.parameters())
print("nn.Linear(3, 5) 파라미터:", total, "= 3×5 + 5")

## 5. 여러 개를 한 번에: `(B, T, C)` 입력

`nn.Linear`는 **마지막 축에만** 작용합니다. 앞의 축(batch, token)은 그대로 둡니다.
Day 2에서 본 "Token 개수는 그대로, 벡터 길이만 바뀐다"와 같은 이야기예요.

In [ ]:
B, T, C = 2, 4, 8
x = torch.randn(B, T, C)

proj = nn.Linear(C, 16)
out = proj(x)

print("입력:", tuple(x.shape))
print("출력:", tuple(out.shape), " ← 마지막 축만 8 → 16")

## 6. LLM과의 연결: Q, K, V는 전부 Linear Layer

Day 2에서 `Q = X @ Wq`라고 했던 그 `Wq`가 실제 GPT 코드에서는 이렇게 생겼습니다.

In [ ]:
head_size = 16

query = nn.Linear(C, head_size, bias=False)
key   = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

q = query(x)
k = key(x)
v = value(x)
print("q:", tuple(q.shape), " k:", tuple(k.shape), " v:", tuple(v.shape))

> 💡 Attention에서는 보통 `bias=False`로 둡니다. Q·K의 Dot Product로 "관련도"를 재는데, Bias가 크게 도움이 되지 않아서 생략하는 경우가 많아요.
>
> GPT 안에서 Linear Layer가 쓰이는 곳: **Q/K/V 만들기, Attention 출력 합치기, FFN (Day 17), 마지막에 Vocabulary 점수 만들기**. 거의 모든 곳이에요.

## 7. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** `nn.Linear(4, 6)`에 대해 실행하기 **전에** 예측해서 적으세요.

In [ ]:
layer = nn.Linear(4, 6)

weight_shape = "?"   # TODO: 튜플로. 3-2의 주의할 점을 떠올려보세요
bias_shape   = "?"   # TODO
num_params   = "?"   # TODO: 숫자 하나

assert weight_shape == tuple(layer.weight.shape), "weight shape를 다시 생각해보세요"
assert bias_shape == tuple(layer.bias.shape), "bias shape를 다시 생각해보세요"
assert num_params == sum(p.numel() for p in layer.parameters()), "파라미터 개수를 다시 세어보세요"
print("Q1 정답!")

**Q2.** `nn.Linear`를 쓰지 않고, `weight`와 `bias`만으로 같은 계산을 하는 함수를 완성하세요.
`weight`는 PyTorch 방식 `(out, in)`으로 들어옵니다.

In [ ]:
def my_linear(x, weight, bias):
    return None  # TODO

layer = nn.Linear(4, 6)
x = torch.randn(4)

assert torch.allclose(my_linear(x, layer.weight, layer.bias), layer(x))
x_batch = torch.randn(2, 5, 4)
assert torch.allclose(my_linear(x_batch, layer.weight, layer.bias), layer(x_batch), atol=1e-6)
print("Q2 정답!")

**Q3.** 입력 `(B=2, T=10, C=32)`를 받아서 출력이 `(2, 10, 128)`이 되도록 Linear Layer를 만드세요.
그 다음 다시 `(2, 10, 32)`로 되돌리는 Layer도 만드세요. (Day 17 FFN의 뼈대예요!)

In [ ]:
x = torch.randn(2, 10, 32)

up   = None  # TODO
down = None  # TODO

h = up(x)
out = down(h)
assert tuple(h.shape) == (2, 10, 128)
assert tuple(out.shape) == (2, 10, 32)
print("Q3 정답!", tuple(h.shape), "→", tuple(out.shape))

**Q4.** GPT의 마지막 단계는 각 Token 벡터 `(C=64)`를 **Vocabulary 크기(65)만큼의 점수**로 바꾸는 Linear Layer입니다.
이 Layer를 만들고, 파라미터 개수를 계산하세요. (bias 있음)

In [ ]:
vocab_size, n_embd = 65, 64

lm_head = None          # TODO
my_count = "?"          # TODO: 손으로 계산한 숫자

x = torch.randn(1, 8, n_embd)
logits = lm_head(x)
assert tuple(logits.shape) == (1, 8, vocab_size)
assert my_count == sum(p.numel() for p in lm_head.parameters()), "파라미터 개수를 다시 세어보세요"
print("Q4 정답! logits:", tuple(logits.shape))

## 8. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Weight는 누가 정하는가?
2. Bias는 왜 필요한가?
3. `nn.Linear(3, 5)`에서 3과 5는 무엇을 의미하는가? 내부 `weight`의 shape는?
4. `(B, T, C)` 입력에 `nn.Linear(C, H)`를 적용하면 결과 shape는? 어느 축이 바뀌는가?
5. Day 2의 `Q = X @ Wq`와 오늘 배운 Linear Layer는 어떤 관계인가?

**내 답:**

1.
2.
3.
4.
5.